In [ ]:
# ============================================================
# WFGY RAG 16 Problem Map · Colab Wave 0 MVP (Single Cell)
# ============================================================
#
# This notebook:
# - Loads wfgy_problem_catalog_v1.json
# - Loads wfgy_debug_packet_v1.json
# - Extracts example_packet
# - Prints a structured Wave 0 triage summary
#
# No LLM is used.
# Purpose: prove JSON specs are machine-readable and tool-friendly.
# ============================================================

import requests

CATALOG_URL = "https://raw.githubusercontent.com/onestardao/WFGY/main/ProblemMap/specs/wfgy_problem_catalog_v1.json"
PACKET_URL = "https://raw.githubusercontent.com/onestardao/WFGY/main/ProblemMap/specs/wfgy_debug_packet_v1.json"


def load_json(url: str) -> dict:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()


def indent_block(text: str, prefix: str = "    ") -> str:
    if not text:
        return ""
    return "\n".join(prefix + line for line in text.splitlines())


def summarize_case(case_obj: dict, catalog: dict) -> str:
    lines = []

    lines.append("=== WFGY RAG 16 Problem Map · Wave 0 Triage Summary ===")
    lines.append("")

    env = case_obj.get("environment", {}) or {}
    lines.append(f"System   : {env.get('system', 'unknown-system')}")
    lines.append(f"Project  : {env.get('project', 'unknown-project')}")
    lines.append(f"Run ID   : {env.get('run_id', 'unknown-run-id')}")
    if env.get("run_url"):
        lines.append(f"Run URL  : {env.get('run_url')}")
    lines.append("")

    question = (case_obj.get("Q") or "").strip()
    if question:
        lines.append("Question:")
        lines.append(indent_block(question))
        lines.append("")

    metrics = case_obj.get("metrics", {}) or {}
    if metrics:
        lines.append("Metrics:")
        for k, v in metrics.items():
            lines.append(f"  - {k}: {v}")
        lines.append("")

    labels = case_obj.get("labels", {}) or {}
    lines.append("Labels:")
    lines.append(f"  primary   : {labels.get('primary', [])}")
    lines.append(f"  secondary : {labels.get('secondary', [])}")
    lines.append(f"  modes     : {labels.get('modes', [])}")
    lines.append(f"  types     : {labels.get('types', [])}")
    if labels.get("confidence") is not None:
        lines.append(f"  confidence: {labels.get('confidence')}")
    lines.append("")

    wave0_ids = catalog.get("wave0_scope", {}).get("modes", [])
    all_modes = catalog.get("modes", [])
    mode_lookup = {m["id"]: m for m in all_modes if isinstance(m, dict)}

    lines.append("Wave 0 Modes In Scope (1, 2, 5, 8):")
    for mid in wave0_ids:
        meta = mode_lookup.get(mid, {})
        lines.append(f"  - {mid} {meta.get('code','')} {meta.get('short_name','')}")
    lines.append("")

    selected_modes = labels.get("modes", [])
    if selected_modes:
        lines.append("Selected Modes For This Case:")
        for mid in selected_modes:
            meta = mode_lookup.get(mid, {})
            lines.append(f"  - {mid} {meta.get('code','')} {meta.get('short_name','')}")
        lines.append("")

    fix_plan = case_obj.get("fix_plan", {}) or {}
    if fix_plan:
        lines.append("Fix Plan:")
        if fix_plan.get("target_modes"):
            lines.append(f"  target_modes  : {fix_plan.get('target_modes')}")
        if fix_plan.get("selected_modes"):
            lines.append(f"  selected_modes: {fix_plan.get('selected_modes')}")

        if fix_plan.get("fixes"):
            lines.append("  fixes:")
            for f in fix_plan["fixes"]:
                lines.append(f"    - {f}")

        if fix_plan.get("verification_tests"):
            lines.append("  verification_tests:")
            for t in fix_plan["verification_tests"]:
                lines.append(f"    - {t}")
        lines.append("")

    triage_notes = (case_obj.get("triage_notes") or "").strip()
    if triage_notes:
        lines.append("Triage Notes:")
        lines.append(indent_block(triage_notes))
        lines.append("")

    return "\n".join(lines)


print("Loading WFGY JSON specs...")
catalog = load_json(CATALOG_URL)
packet_spec = load_json(PACKET_URL)
example_case = packet_spec.get("example_packet", {}) or {}

print("Catalog URL :", CATALOG_URL)
print("Packet URL  :", PACKET_URL)
print("")

print(summarize_case(example_case, catalog))

print("=== Wave 0 JSON MVP Execution Complete ===")